# 30 · The drift check: is the model still seeing the hospital it learned?

**Who this is for:** you, after notebook 21 (batch scoring) and notebook 04 (the drift scenarios, offline).
**Runs on:** Databricks serverless (costs credits: keep it short, detach when done). The `drift_check` job must have run in the catalog you pick: staging runs it after every merge; prod via **prod-run** with *Check for drift* ticked.

**By the end you will be able to:**
- read a drift check: its overall level, and which signals breached where
- compute a PSI by hand and say what "act" means for it
- explain why the check compares with the champion's **own** data (training and validation blocks)
- explain why AUPRC and Brier are judged with a margin for chance, and why they lag a day
- say, for each kind of drift, which signal shows it and what the right response is (retrain or not)

The job is `databricks/resources/drift_check.job.yml`; the logic is `heavden.monitoring.drift_check`, tested locally (`tests/test_drift_check.py`). The response to each signal is in the *Drift Incident Runbook* (`docs/corpus/drift-incident-runbook.md`).

## 1. Setup

The check judges one model (prod: `@champion`; dev and staging: the newest version) and scores everything it compares with that model itself, so this notebook loads it too, with LightGBM installed once per session.

In [ ]:
%pip install lightgbm==4.6.0 --quiet

In [ ]:
import sys

# Find the repo folder this notebook was deployed into, and import the real code from it.
notebook_path = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
repo_root = "/Workspace" + notebook_path.split("/notebooks/databricks/")[0]
sys.path.insert(0, f"{repo_root}/databricks/src")

import mlflow
import pandas as pd

from heavden.ml import batch_scoring, train
from heavden.ml.training import to_utc
from heavden.monitoring import drift_check, drift_metrics

dbutils.widgets.text("catalog", "heavden_staging")
CATALOG = dbutils.widgets.get("catalog")
spark.conf.set("spark.sql.session.timeZone", "UTC")
mlflow.set_registry_uri("databricks-uc")
alias = "champion" if CATALOG == "heavden_prod" else "latest"
model, threshold, version = batch_scoring.load_registered_model(
    f"{CATALOG}.ml.deterioration_risk", alias
)
print(repo_root, CATALOG, f"model v{version} ({alias})")

## 2. The checks so far

One row per run of the job: when, which episode had just been loaded (empty for a normal run), which model version it judged (it scores everything it compares with that version; the prediction log can hold older versions' scores, because each hour is scored once), the windows it looked at, and the result. `status` is the worst level of any signal: `stable`, `warn` or `act`.

In [ ]:
display(
    spark.sql(f"""
    SELECT checked_at, episode, model_version, window_start, window_end, labels_through,
           status, n_act, n_warn, breaches
    FROM {CATALOG}.monitoring.drift_checks ORDER BY checked_at DESC
""")
)

## 3. Every signal of the latest check

`value` is what the level is judged on; `observed` is the measured quantity when the two differ (AUPRC itself for an AUPRC drop, a field's missing share for its change); `reference` is what it's compared with; `n` is the sample (rows, readings, or escalations for AUPRC and Brier). Slices with too little data say `not enough data` rather than guess.

In [ ]:
latest = spark.sql(f"""
    SELECT * FROM {CATALOG}.monitoring.drift_metrics
    WHERE check_id = (SELECT max_by(check_id, checked_at) FROM {CATALOG}.monitoring.drift_checks)
""").toPandas()
rank = {"act": 0, "warn": 1, "stable": 2, "not enough data": 3}
columns = ["signal", "feature", "site_id", "value", "observed", "reference", "n", "level"]
display(latest.assign(rank=latest["level"].map(rank)).sort_values(["rank", "signal"])[columns])

## 4. Inputs, feature by site

The same PSI values as a grid. A sensor fault lights up **one cell** (one feature at one site); a new kind of patient moves **a row** (a feature at every site); a protocol change moves **nothing here** at all, because the vital signs haven't changed, only what staff do about them.

In [ ]:
psi = latest[latest["signal"] == "feature_psi"]
display(psi.pivot(index="feature", columns="site_id", values="value").round(3).reset_index())

## 5. A PSI by hand

PSI (population stability index) cuts the reference data into 10 bins holding the same share each, then asks how the new data spreads over the same bins: `sum((new - old) * ln(new / old))`. Below 0.1 is stable, 0.1-0.25 a moderate shift, above 0.25 a major one. Unlike a KS test's p-value it measures **how big** the shift is, so tens of thousands of rows don't make a trivial difference look alarming.

The reference is the champion's **own** data, its training and validation blocks: what it was fitted and calibrated on. (Training alone is 18 hours in staging, too short: normal days then read as drift.)

In [ ]:
training_set = to_utc(
    spark.read.table(f"{CATALOG}.gold.training_set").toPandas(), ("prediction_ts", "label_known_at")
)
features = to_utc(
    spark.read.table(f"{CATALOG}.gold.patient_hour_features").toPandas(), ("prediction_ts",)
)
days = (1, 0.5) if CATALOG != "heavden_prod" else (9, 2)  # the bundle's train and valid days
reference = pd.concat(train.time_split(training_set, train.Split(*days))[:2])
last_hour = features["prediction_ts"].max()
latest_day = features[features["prediction_ts"] > last_hour - drift_check.WINDOW]

FEATURE, SITE = "spo2_mean_1h", "SITE_B"
old = reference.loc[reference["site_id"] == SITE, FEATURE].to_numpy()
new = latest_day.loc[latest_day["site_id"] == SITE, FEATURE].to_numpy()
edges = drift_metrics.baseline_bins(old)
bins = pd.DataFrame(
    {
        "from": edges[:-1].round(2),
        "to": edges[1:].round(2),
        "reference share": drift_metrics.bin_shares(old, edges).round(3),
        "latest day share": drift_metrics.bin_shares(new, edges).round(3),
    }
)
display(bins)
value = drift_metrics.psi(old, new)
print(f"PSI {value:.3f}: {drift_metrics.psi_level(value)}")

## 6. Performance, with a margin for chance

AUPRC and Brier need outcomes, which arrive about 12 hours late (the 6-hour window, then 6 hours until an escalation is recorded). So this signal looks at the latest **48 labelled hours**, which end about 12 hours before the data does.

Two days hold only about 20 escalations, so the AUPRC of a two-day window swings by a third either way on normal days (measured offline on prod's history). Judged on the raw number, normal days would raise "act". The check therefore resamples whole stays 200 times (a bootstrap) and uses the drop it can be 95% sure of: from the **top** of the AUPRC interval. Below 10 escalations it reports `not enough data`.

In [ ]:
newest = training_set["prediction_ts"].max()
labelled = training_set[training_set["prediction_ts"] > newest - drift_check.LABEL_WINDOW]
labelled = labelled.assign(risk=model.predict_proba(labelled)[:, 1])  # the judged model's scores
boot = drift_check.bootstrap_scores(labelled, n_boot=200, seed=0)
perf = latest[latest["signal"].isin(["auprc_drop", "brier_rise"])].set_index("signal")
drop = perf.loc["auprc_drop"]
print(f"labelled hours {len(labelled):,}, escalations {drop['n']}")
print(
    f"AUPRC {drop['observed']:.3f}, 90% of resamples between "
    f"{boot['auprc'].quantile(0.05):.3f} and {boot['auprc'].quantile(0.95):.3f}; "
    f"validation {drop['reference']:.3f}"
)
print(f"drop we're 95% sure of: {drop['value']:.0%} ({drop['level']})")

## 7. From signal to decision

The check reports; a person decides (Plan.md §10.4). Measured offline on prod's history with its two episodes:

| What happened | What the check shows | Right response |
|---|---|---|
| Normal days | nothing at act; at most a warn (Site B SpO2 PSI up to 0.15) | none |
| SpO2 sensors read low at Site B | act at Site B only: SpO2 PSI 1.2, NEWS2 PSI, prediction JS, alert load 3.3; performance unchanged | fix the sensors, **don't retrain** |
| Escalation protocol changed | inputs stable; act on AUPRC (-44%) and Brier (+36%), once outcomes are known | **retrain** on data after the change |
| Firmware drops `motion`, adds `etco2` (staging) | act on those fields' missing shares | a pipeline fix, not a model fix |

The pattern to remember: **where** (one site or all) and **what** (inputs, outcomes or the payload) point to the cause before any model is touched.

Staging is a plumbing check: its model trains on one day, so its reference is short and its numbers are noisier than prod's.

In [ ]:
# Try it yourself: 1. Recompute the PSI in section 5 with 5 bins (n_bins=5). Does the level change?

In [ ]:
# Try it yourself: 2. Which site and day had the highest alert load? (gold.alerts_fact)

In [ ]:
# Try it yourself: 3. After an episode is loaded, compare its check with the one before it.

**Pitfall: drift is not a reason to retrain by itself.** A sensor fault moves the inputs a lot, yet retraining would teach the model to trust broken readings.

**Pitfall: judging performance on a handful of events.** With ~10 escalations a day, one day's AUPRC is mostly noise; use enough outcomes and a margin for chance.

**Next:** alerting (an email, and the GitHub retrain request) when a check reaches act.